# Animal Face Generation — public reproduction notebook

A shared VQ representation, a DDPM-trained U-Net, and an autoregressive Transformer on AFHQv2 at 128×128.

This output-free notebook preserves the recorded run's code and configuration: **100 tokenizer epochs, 200 diffusion epochs and 200 Transformer epochs**, FP32, global effective batch 64, and full evaluation. All warm-up values are **0**, so the current scheduler keeps both generator learning rates constant. This matches the executed experiment, which differed from the intended five-epoch warm-up and cosine schedule. Executing this notebook starts a new reproduction attempt; the [technical report](../reports/technical-report.md) contains the previously measured results and limitations. The recorded Transformer produced recognizable faces, while diffusion produced abstract patterns.

Device priority is **MPS → two CUDA GPUs → one CUDA GPU → CPU**. With two CUDA GPUs, the tokenizer trains across both, then the priors train concurrently on separate GPUs. Single-device systems use sequential workers. The recorded batch settings completed on the original two-L40S host; a different device needs its own memory check.

Use RunPod's existing **Python 3** kernel. Python 3.13 is preferred; 3.12 is supported. Dependencies are installed in the explicit cell below. **Run All installs dependencies, downloads missing assets and starts training plus full evaluation.** Review the configuration cells first. A fresh-run cell always allocates another numbered run; existing dataset and pretrained-weight caches remain reusable.

Each operation, plot and sampling variant has its own cell. Heavy stages use isolated workers; no custom package or experiment CLI is required. The original output-bearing notebook is retained locally and is excluded from the public submission.


## 1. Select the project root

Use the project folder as the working directory so normal `src` imports work. The same cell works when the kernel starts in `notebooks/` or the project folder.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
PROJECT_ROOT = Path(".")

## Install dependencies

Use the existing Python 3.12 or 3.13 notebook kernel. Run the installation cell below once; no virtual environment creation or kernel registration is required. It uses standard pip with this kernel's Python.

The revised requirements keep the scientific package pins and accept compatible existing dependencies, including Debian's `pyparsing` 3.1.1. Jupyter server/kernel installation is excluded. Stop any older pip installation before starting this cell. If installation replaces packages already loaded in memory, restart the current kernel and run from the top while skipping the completed installation cell.

Dataset and pretrained-model downloads occur later in separate cells. The README describes the CUDA 13 driver requirement.


In [ ]:
import subprocess
import sys


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])


## 2. Import the project functions

In [ ]:
from dataclasses import asdict, replace  # noqa: F401 — explicit analysis revisions

from IPython.display import display
from PIL import Image

from src.config import (
    AutoregressiveConfig,
    DataConfig,
    DiffusionConfig,
    EvaluationConfig,
    ExperimentConfig,
    RuntimeConfig,
    TokenizerConfig,
    TrainingConfig,
)
from src.inspection import describe_model
from src.launcher import ExperimentLauncher
from src.reporting import (
    plot_comparison,
    plot_generation_process,
    plot_training_history,
    show_ddpm_examples,
    show_generated_grid,
    show_nearest_training,
    show_reconstructions,
    show_training_samples,
)
from src.runs import create_run, open_run
from src.sample import load_generator
from src.tracking import launch_tensorboard  # noqa: F401 — optional final cell


## 3. Define configurations

Each model has a separate editable cell. The published defaults reproduce the recorded **100 / 200 / 200 epochs** and **zero warm-up**. Zero warm-up means constant generator learning rates in this implementation. The intended five-epoch warm-up followed by cosine decay was not used in the recorded run.

For a **one-epoch pipeline smoke test**, set all three epoch counts to **1**, retain warm-up **0**, and create a fresh run. This tests execution rather than converged image quality. A future experiment following the original full-budget protocol would use 100 / 200 / 200 epochs with **five warm-up epochs for both generators** and zero for the tokenizer; it must be reported separately. Change epochs or schedules before creating a run, and keep the two generators' training exposure equal. Warm-up must be shorter than training.

The L40S settings use **FP32, eager execution and global effective batch 64**. Tokenizer microbatch **8 is per GPU**; two GPUs do not double the global batch. Diffusion and Transformer each use microbatch **32**. `wild` is the upstream folder name; its display label is wildlife.

Image preparation uses eight threads; each worker uses four CPU threads and `num_workers=0`. Reconstruction and token caching use inference batch **64**, generation uses batch **16**, and feature extraction uses batch **32**. Benchmark batches remain **1 and 8**, with three warm-ups and thirty measured repetitions. These settings completed on the recorded L40S host; fit and speed are not guaranteed on another host. Batch changes can alter floating-point execution and sampling streams and require a fresh run.

For MPS or a smaller device, set the visible configuration cells to these conservative starting values before creating a fresh run: tokenizer microbatch **4**, generator microbatch **16**, tokenizer inference batch **32**, generation batch **8**, and feature batch **8**. Memory fit still needs verification. The recorded diffusion failure and Transformer overfitting should be diagnosed before choosing additional training budgets; see the [report](../reports/technical-report.md).


In [ ]:
data_config = DataConfig(
    url="https://www.dropbox.com/s/vkzjokiwof5h8w6/afhq_v2.zip?dl=1",
    classes=("cat", "dog", "wild"),
    display_classes=("cat", "dog", "wildlife"),
    image_size=128,
    validation_fraction=0.10,
    split_seed=1001,
    expected_total=15803,
)

In [ ]:
tokenizer_config = TokenizerConfig(
    channels=(64, 128, 256), residual_blocks=2, codebook_size=512, embedding_dim=64,
    commitment_weight=0.25, perceptual_weight=0.1,
    training=TrainingConfig(
        epochs=100, seed=2027, microbatch=8, effective_batch=64,
        learning_rate=2e-4, weight_decay=0.0, betas=(0.9, 0.999),
        epsilon=1e-8, gradient_clip=1.0, warmup_epochs=0,
    ),
)

In [ ]:
diffusion_config = DiffusionConfig(
    channels=(64, 128, 256), residual_blocks=2, attention_heads=4,
    attention_sizes=(8, 4), time_embedding_dim=256,
    timesteps=1000, beta_start=0.0001, beta_end=0.02,
    ema_decay=0.999, validation_seed=4001,
    training=TrainingConfig(
        epochs=200, seed=2027, microbatch=32, effective_batch=64,
        learning_rate=1e-4, weight_decay=0.01, betas=(0.9, 0.999),
        epsilon=1e-8, gradient_clip=1.0, warmup_epochs=0, minimum_lr_fraction=0.1,
    ),
)

In [ ]:
autoregressive_config = AutoregressiveConfig(
    layers=6, width=384, heads=6, feedforward_dim=1536, dropout=0.1,
    training=TrainingConfig(
        epochs=200, seed=2027, microbatch=32, effective_batch=64,
        learning_rate=3e-4, weight_decay=0.01, betas=(0.9, 0.999),
        epsilon=1e-8, gradient_clip=1.0, warmup_epochs=0, minimum_lr_fraction=0.1,
    ),
)

In [ ]:
evaluation_config = EvaluationConfig(
    max_per_class=500, sampling_seed=5001, reference_seed=5002, metric_seed=5003,
    ddim_steps=(25, 50, 100), temperatures=(0.8, 1.0, 1.2),
    generation_batch=16, feature_batch=32,
    kid_subsets=100, kid_class_subset=100, kid_overall_subset=500,
    precision_recall_k=3, benchmark_batches=(1, 8),
    benchmark_warmups=3, benchmark_repetitions=30, grid_per_class=12,
)

In [ ]:
runtime_config = RuntimeConfig(
    device="auto", num_workers=0, cpu_threads=4, io_workers=8,
    tokenizer_inference_batch=64,
)

In [ ]:
experiment_config = ExperimentConfig(
    data=data_config, tokenizer=tokenizer_config, diffusion=diffusion_config,
    autoregressive=autoregressive_config, evaluation=evaluation_config,
    runtime=runtime_config,
)

## Inspect hardware

The L40S preset works with one or two visible GPUs. The first two visible CUDA GPUs are selected by UUID; additional GPUs remain unused. Hardware inspection loads no model and reports CUDA build/driver compatibility before training. Models, losses and optimizer states remain FP32; AMP and TF32 are disabled.


In [ ]:
launcher = ExperimentLauncher(PROJECT_ROOT)


In [ ]:
hardware = launcher.inspect_hardware(requested=runtime_config.device)


In [ ]:
display(hardware)


## 4. Select fresh-run or explicit resume

`None` creates a fresh run. To resume, replace it with the existing name, such as `"run_001"`. Save the notebook before creating a run so the source snapshot includes your controls.

In [ ]:
RUN_NAME = None

## 5. Create or open the run

Re-executing this cell with `RUN_NAME=None` creates the next run. Resume verifies source, dependencies, backend, GPU topology and resolved configuration. Use a **fresh run** for this L40S preset, changed batches, changed hardware or changed epochs; earlier runs remain preserved. Shared preparation is exclusive; only distinct generator workers may train concurrently.


In [ ]:
run = (
    create_run(PROJECT_ROOT, experiment_config, hardware=hardware)
    if RUN_NAME is None
    else open_run(PROJECT_ROOT, RUN_NAME, config=experiment_config, hardware=hardware)
)


## 6. Download the authors' AFHQ v2 archive

This is an explicit network operation. AFHQ is supplied for noncommercial research under CC BY-NC 4.0; see the README for provenance and terms. Raw images remain in this project's cache.

In [ ]:
acquisition = launcher.run_action("download_dataset", run.path.name)


## 7. Validate the actual inventory

Inspect every file for corruption and hash exact pixels/bytes. Upstream held-out images are read only for integrity checks here, not visual selection or quality measurement.

In [ ]:
inventory = launcher.run_action("validate_inventory", run.path.name)


## 8. Create or verify the split manifest

Ten percent per upstream training class becomes development validation. The upstream `test` partition remains the final test set (legacy `val` layouts are also recognized). Original archive split names are preserved. This records the balanced evaluation count before training and rejects exact cross-split duplicates.

In [ ]:
data_manifest = launcher.run_action("prepare_data", run.path.name)


## 9. Inspect training examples

In [ ]:
display(show_training_samples(run))

## 10. Acquire frozen pretrained weights

This explicit cell downloads VGG-16 and compatible Inception weights and copies installed LPIPS calibration weights. Their complete hashes are recorded. Loaders never download automatically.

In [ ]:
weights = launcher.run_action("acquire_weights", run.path.name)


## 11. Display tokenizer architecture

In [ ]:
print(describe_model(run, "tokenizer"))

## 12. Train the tokenizer

The recorded default is **100 epochs**; checkpoints commit after each completed epoch. Two CUDA GPUs use DDP with global effective batch 64. MPS, one CUDA GPU and CPU use one worker. Selection minimizes development-validation L1 + 0.1 LPIPS, with earliest ties. A completed stage is verified and reused.

The next code cell preserves the original Pod's explicit `NCCL_DEBUG="INFO"` and `NCCL_P2P_DISABLE="1"` settings. Disabling P2P allowed that two-L40S Pod to use another NCCL transport after a startup stall. This is **Pod-specific**, not a universal performance default. Review or remove that override before launching on another host; the public code retains it to document the executed configuration.

Progress and resource observations are saved with worker logs. Elapsed time is reported without timeout warnings. Only new log-tail content is printed. An interrupted partial epoch is repeated on resume.


In [ ]:
os.environ.update(NCCL_DEBUG="INFO", NCCL_P2P_DISABLE="1")

In [ ]:
tokenizer_training = launcher.run_action("train_tokenizer", run.path.name)


## 13. Evaluate validation reconstructions

In [ ]:
reconstruction_metrics = launcher.run_action("evaluate_reconstructions", run.path.name)


## 14. Display reconstruction examples

In [ ]:
display(show_reconstructions(run))

## 15. Freeze the selected tokenizer

The full tokenizer budget and diagnostic evaluation must be complete. Both priors will record this same best-validation checkpoint hash.

In [ ]:
tokenizer_identity = launcher.run_action("freeze_tokenizer", run.path.name)


## 16. Cache training and validation tokens

Only training tokens define the scalar embedding mean and standard deviation. No test tokens are cached for model selection.

In [ ]:
token_cache = launcher.run_action("cache_tokens", run.path.name)


## 17. Display diffusion architecture

In [ ]:
print(describe_model(run, "diffusion"))

## 18. Start diffusion

The recorded default is **200 epochs with zero warm-up and constant learning rate**. Validation uses fixed noise/timesteps; raw and EMA weights are saved. Final EMA weights are the predefined primary evaluation choice. This cell starts a worker and returns so the Transformer can start next.


In [ ]:
diffusion_worker = launcher.start_training(run.path.name, "diffusion")


## 19. Display Transformer architecture

In [ ]:
print(describe_model(run, "autoregressive"))

## 20. Start the Transformer

The recorded default is **200 epochs with zero warm-up and constant learning rate**, matching the diffusion model's image exposure and global effective batch. With two CUDA GPUs this worker runs beside diffusion on its own GPU. Otherwise it queues behind diffusion and starts in the wait cell. Final-epoch weights are the primary evaluation choice; best-validation weights are retained separately.


In [ ]:
autoregressive_worker = launcher.start_training(run.path.name, "autoregressive")


## Inspect worker status

Status reads do not start queued work.


In [ ]:
display(launcher.status())


## Wait for both generator workers

On a single device this cell starts the queued Transformer after diffusion finishes. On two CUDA GPUs it waits for both concurrent workers. Only continue to evaluation after success. `launcher.cancel_all()` explicitly stops workers if needed.


In [ ]:
generator_training = launcher.wait([diffusion_worker, autoregressive_worker])


## 21. Plot each model's own history

Loss magnitudes across these objectives are not comparable quality scores.

In [ ]:
display(plot_training_history(run, "tokenizer"))

In [ ]:
display(plot_training_history(run, "diffusion"))

In [ ]:
display(plot_training_history(run, "autoregressive"))

## 22. Freeze checkpoint hashes and evaluation settings

Run this before opening the final reference or inspecting generated comparisons. Repeating identical settings reuses the analysis. Explicit changed evaluation settings allocate `analysis_002`, etc.; they never replace prior results. For later revisions use `replace(run.config.evaluation, ...)` here while preserving the run's original training configuration.

In [ ]:
analysis = launcher.run_action("freeze_evaluation", run.path.name, evaluation_config=asdict(evaluation_config))


## 23. Prepare the balanced held-out reference

In [ ]:
reference = launcher.run_action("prepare_reference", run.path.name, analysis=analysis)


## 24. Generate each fixed setting separately

All six measured settings use the same requested class counts. Initial Gaussian draws are shared across DDIM step settings through fixed per-class/batch seeds. Matching seed integers across model families do not match trajectories.

In [ ]:
launcher.run_action("generate_samples", run.path.name, family="diffusion", setting="ddim_50", analysis=analysis)


In [ ]:
launcher.run_action("generate_samples", run.path.name, family="autoregressive", setting="temperature_1.0", analysis=analysis)


In [ ]:
launcher.run_action("generate_samples", run.path.name, family="diffusion", setting="ddim_25", analysis=analysis)


In [ ]:
launcher.run_action("generate_samples", run.path.name, family="diffusion", setting="ddim_100", analysis=analysis)


In [ ]:
launcher.run_action("generate_samples", run.path.name, family="autoregressive", setting="temperature_0.8", analysis=analysis)


In [ ]:
launcher.run_action("generate_samples", run.path.name, family="autoregressive", setting="temperature_1.2", analysis=analysis)


### DDPM illustration

A separate small grid of original DDPM samples is illustrative only; it is not mixed into the six measured settings.

In [ ]:
ddpm_examples = launcher.run_action("generate_ddpm_examples", run.path.name, class_name="cat", count=4, seed=6001, analysis=analysis)


In [ ]:
display(show_ddpm_examples(run, analysis, class_name="cat", seed=6001, count=4))

## 25. Evaluate each generated set separately

KID macro-average across classes is primary; overall and individual-class KID plus precision/recall are diagnostics. Features are identical across metrics, statistics run on CPU, and negative unbiased KID estimates are retained.

In [ ]:
launcher.run_action("evaluate_samples", run.path.name, analysis=analysis, setting="ddim_50")


In [ ]:
launcher.run_action("evaluate_samples", run.path.name, analysis=analysis, setting="temperature_1.0")


In [ ]:
launcher.run_action("evaluate_samples", run.path.name, analysis=analysis, setting="ddim_25")


In [ ]:
launcher.run_action("evaluate_samples", run.path.name, analysis=analysis, setting="ddim_100")


In [ ]:
launcher.run_action("evaluate_samples", run.path.name, analysis=analysis, setting="temperature_0.8")


In [ ]:
launcher.run_action("evaluate_samples", run.path.name, analysis=analysis, setting="temperature_1.2")


## 26. Benchmark each setting separately

Training workers have finished. Every benchmark runs serially in an isolated process on the same first device. Each setting measures batches 1 and 8: a separate first call, three warm-ups and thirty synchronized repetitions. Generation includes decoder and CPU uint8 conversion; disk writes, plotting and metric extraction are excluded. First-call measurements use newly loaded wrappers.


In [ ]:
launcher.run_action("benchmark_generation", run.path.name, family="diffusion", setting="ddim_50", analysis=analysis)


In [ ]:
launcher.run_action("benchmark_generation", run.path.name, family="autoregressive", setting="temperature_1.0", analysis=analysis)


In [ ]:
launcher.run_action("benchmark_generation", run.path.name, family="diffusion", setting="ddim_25", analysis=analysis)


In [ ]:
launcher.run_action("benchmark_generation", run.path.name, family="diffusion", setting="ddim_100", analysis=analysis)


In [ ]:
launcher.run_action("benchmark_generation", run.path.name, family="autoregressive", setting="temperature_0.8", analysis=analysis)


In [ ]:
launcher.run_action("benchmark_generation", run.path.name, family="autoregressive", setting="temperature_1.2", analysis=analysis)


## 27. Produce individual comparison figures

In [ ]:
display(show_generated_grid(run, analysis, "ddim_50"))

In [ ]:
display(show_generated_grid(run, analysis, "temperature_1.0"))

In [ ]:
display(show_generated_grid(run, analysis, "ddim_25"))

In [ ]:
display(show_generated_grid(run, analysis, "ddim_100"))

In [ ]:
display(show_generated_grid(run, analysis, "temperature_0.8"))

In [ ]:
display(show_generated_grid(run, analysis, "temperature_1.2"))

In [ ]:
display(plot_comparison(run, analysis, metric="kid"))

In [ ]:
display(plot_comparison(run, analysis, metric="overall_kid"))

In [ ]:
display(plot_comparison(run, analysis, metric="precision"))

In [ ]:
display(plot_comparison(run, analysis, metric="recall"))

In [ ]:
display(plot_comparison(run, analysis, metric="quality_speed"))

### Nearest training-image diagnostics

Fixed generated examples are compared to their nearest training Inception features. Similarity or lack of an obvious match does not prove memorization or non-memorization.

In [ ]:
display(show_nearest_training(run, analysis, "ddim_50"))

In [ ]:
display(show_nearest_training(run, analysis, "temperature_1.0"))

### Generation processes

Each creation and display is separate. Diffusion panels decode snapped snapshots for illustration only; sampling itself never quantizes intermediate steps. Transformer panels color actual token prefixes, with unknown positions dark.

In [ ]:
launcher.run_action("generate_diffusion_process", run.path.name, class_name="cat", seed=6001, steps=50, analysis=analysis)


In [ ]:
display(plot_generation_process(run, analysis, family="diffusion", class_name="cat", seed=6001, steps=50))

In [ ]:
launcher.run_action("generate_autoregressive_process", run.path.name, class_name="cat", seed=6001, analysis=analysis)


In [ ]:
display(plot_generation_process(run, analysis, family="autoregressive", class_name="cat", seed=6001))

## 28. Write the technical report

The report uses saved observations and discloses missing stages; it does not invent a winner or infer training-seed uncertainty.

In [ ]:
report_path = launcher.run_action("write_report", run.path.name, analysis=analysis)


## 29. Load saved models for inference

These calls load configuration, prior weights and the shared codec. No raw dataset or token cache is required. In a matching environment, `run` can also be obtained by opening an explicit completed run. Change the requested class to cat, dog or wildlife. Loading and generation are separate operations; generation returns BCHW uint8 CPU images.

The current loader enforces the run's original source, environment, backend and topology. It does not provide portable loading of the saved L40S run on an MPS Mac. The locally downloaded historical run is also missing the snapshot's `.python-version`; recovery must preserve its recorded identity. A fresh complete run creates its own artifacts. See the [README](../README.md) for recovery constraints.


In [ ]:
inference_generator = load_generator(run, family="diffusion")

In [ ]:
inference_images = inference_generator.generate(class_name="wildlife", count=4, steps=50, sampler="ddim", seed=8001)

In [ ]:
display(Image.fromarray(inference_images[0].permute(1, 2, 0).numpy()))

In [ ]:
del inference_generator

In [ ]:
inference_generator = load_generator(run, family="autoregressive")

In [ ]:
inference_images = inference_generator.generate(class_name="dog", count=4, temperature=1.0, seed=8001)

In [ ]:
display(Image.fromarray(inference_images[0].permute(1, 2, 0).numpy()))

In [ ]:
del inference_generator

## 30. Optional TensorBoard dashboard

Uncomment to launch on localhost. JSONL/CSV and checkpoint commits remain the durable measurement records.

In [ ]:
# tensorboard_url = launch_tensorboard(run)

## Interpretation boundary

Equal epochs and effective batches control data exposure, not parameter count, compute or convergence. The compact nonadversarial tokenizer may limit both priors. Reconstruction diagnostics are not mathematical bounds on generative metrics. AFHQ's small holdout and ImageNet-derived feature networks limit metric interpretation. One training seed does not support claims about repeatability or a universal winning model family.

The saved run shows a diffusion-generation failure, Transformer overfitting after its best-validation epoch, low tokenizer codebook usage, and a constant-learning-rate deviation from the intended schedule. Existing held-out results have already been inspected; follow-up tuning should use development validation and be described as exploratory. Read the [technical report](../reports/technical-report.md) before interpreting or extending this experiment.

Research: [VQ-VAE](https://arxiv.org/abs/1711.00937), [DDPM](https://arxiv.org/abs/2006.11239), [Taming Transformers](https://arxiv.org/abs/2012.09841), [Latent Diffusion](https://arxiv.org/abs/2112.10752), [DDIM](https://arxiv.org/abs/2010.02502), [KID](https://arxiv.org/abs/1801.01401). This project adapts these ideas; it does not reproduce their reported results.
